In [ ]:

!pip install transformers datasets sentencepiece accelerate -q

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from datasets import load_dataset
import torch

In [ ]:
dataset = load_dataset("squad")

print(dataset)
print("\nExample:")
print("Context:", dataset["train"][0]["context"][:200])
print("Question:", dataset["train"][0]["question"])

In [ ]:
# Load Valhalla Model and Tokenizer

model_name = "valhalla/t5-base-qg-hl"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Model loaded! ")

In [ ]:
# Select Subset
small_train = dataset["train"].select(range(15000))
small_val = dataset["validation"].select(range(1500))

print(f"Train size: {len(small_train)}")
print(f"Validation size: {len(small_val)}")

In [ ]:
# Preprocess / Tokenize Data
MAX_INPUT = 512
MAX_TARGET = 64

def preprocess(examples):
    inputs = []
    targets = []
    
    for context, question in zip(examples["context"], examples["question"]):

        # Context me quistion ke answer ko highlight krta h
        input_text = "generate question: " + context
        inputs.append(input_text)
        targets.append(question)
    
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length",
    )
    
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length",
    )
    
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# Apply Preprocessing

tokenized_train = small_train.map(
    preprocess,
    batched=True,
    remove_columns=small_train.column_names
)
tokenized_val = small_val.map(
    preprocess,
    batched=True,
    remove_columns=small_val.column_names
)

print("Tokenization done")

In [ ]:
# Training Arguments

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./valhalla-quiz-generator",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=300,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    predict_with_generate=True,
    report_to="none"
)

In [ ]:
# Trainer Setup

from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [ ]:
# train the model

trainer.train()

In [ ]:
# Save Model
model.save_pretrained("./valhalla-quiz-generator")
tokenizer.save_pretrained("./valhalla-quiz-generator")

print("Model 3 saved!")

In [ ]:
#  Test Model
def generate_question(context):
    input_text = "generate question: " + context
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test karo
context = """
Backpropagation is a method to calculate gradients in neural networks 
which helps in updating weights to minimize loss. It works by calculating 
how much each weight contributed to the prediction error and then adjusting 
the weights accordingly using gradient descent.
"""

print("Question:", generate_question(context))